# E3 — Frozen Bentong S2+R1 SVM applied to Pahang outside Bentong

**Analytical status:** independent external evaluation.

This notebook applies the already frozen Bentong `SVM + S2 + R1` model
to independently labelled polygons from Pahang outside Bentong. Pahang
labels are used only after prediction to calculate external accuracy.

This workflow performs **no** Pahang retraining, feature selection,
hyperparameter tuning, threshold adjustment or class merging. The
internally stored class-3 name `Mixed agriculture` is reported as
`Other agriculture`; its numeric encoding remains `3`.


## How to use this notebook

Run the cells from top to bottom in Google Colab. The reconstruction has
four safeguards:

1. it verifies the SHA-256 hashes of the Pahang CSV and frozen model;
2. it locks the 13 S2 predictors and the R1 SVM configuration;
3. it never calls `.fit()` on Pahang data;
4. it checks the reproduced metrics and polygon confusion matrix against
   the completed E3 run from July 2026.

The E2 comparison is descriptive and configuration-matched. It does not
prove that any observed difference was caused by a single source of domain
shift.


## 1. Mount Google Drive and load libraries


In [ ]:
from pathlib import Path
import os


def _find_repository_root(start: Path) -> Path:
    current = start.resolve()
    while current.parent != current:
        if (current / "README.md").exists() and (current / "code").exists():
            return current
        current = current.parent
    raise RuntimeError("Run this notebook from within the repository tree.")


REPO_ROOT = _find_repository_root(Path.cwd())
DATA_ROOT = Path(
    os.environ.get("DURIAN_DATA_ROOT", REPO_ROOT / "private_data")
).expanduser().resolve()
REPO_OUTPUT_ROOT = Path(
    os.environ.get("DURIAN_OUTPUT_ROOT", REPO_ROOT / "outputs" / "runs")
).expanduser().resolve()
REPO_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("Private data root:", DATA_ROOT)
print("Run output root:", REPO_OUTPUT_ROOT)


In [ ]:
import hashlib
import json
import os
import platform
import sys
import time
import warnings
from datetime import datetime, timezone
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
from IPython.display import display
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
)

warnings.filterwarnings('once')
sns.set_theme(style='whitegrid', context='notebook')

print('Python:', platform.python_version())
print('pandas:', pd.__version__)
print('numpy:', np.__version__)
print('scikit-learn:', sklearn.__version__)
print('joblib:', joblib.__version__)


## 2. Paths and locked E3 settings


In [ ]:
DRIVE_ROOT = DATA_ROOT

INPUT_CSV = (
    DRIVE_ROOT / 'Pahang_without_Bentong_S2_pixel_samples_2025_v1.csv'
)
MODEL_BUNDLE_PATH = (
    DRIVE_ROOT / 'SVM_grouped_validation_20260624_run01' /
    'models' / 'svm_final_bundle.joblib'
)
E2_SUMMARY_PATH = (
    DRIVE_ROOT / 'E2_Bentong_fixed_S2_R1_grouped_OOF_20260813_run01' /
    'tables' / 'fixed_R1_feature_group_summary.csv'
)

RUN_NAME = (
    'E3_Bentong_frozen_S2_R1_to_Pahang_external_validation_'
    '20260816_run01'
)
OUTPUT_DIR = REPO_OUTPUT_ROOT / RUN_NAME
TABLE_DIR = OUTPUT_DIR / 'tables'
FIGURE_DIR = OUTPUT_DIR / 'figures'
METADATA_DIR = OUTPUT_DIR / 'metadata'

MAX_PIXELS_PER_SAMPLE = 100
VERIFY_INPUT_HASHES = True
REQUIRE_LOCKED_RESULT_MATCH = True
RESULT_ABSOLUTE_TOLERANCE = 1e-9

EXPECTED_INPUT_CSV_SHA256 = (
    'ae640a1dad129b284619b509602dc564dba3f2390507e13bd2c25f285e4db668'
)
EXPECTED_MODEL_BUNDLE_SHA256 = (
    'a30549049292072af77dda4157144a74a09d50a25575a2d06f405b78271ebc35'
)
EXPECTED_RAW_ROWS = 227_002
EXPECTED_RETAINED_ROWS = 59_975
EXPECTED_SAMPLE_COUNT = 652
EXPECTED_GROUP_COUNT = 268
EXPECTED_RANDOM_SEED = 42
EXPECTED_SKLEARN_VERSION = '1.6.1'

EXPECTED_S2_FEATURES = [
    'B3', 'B4', 'B5', 'B6', 'B7',
    'B8', 'B8A', 'B11', 'B12',
    'NDVI', 'NDRE', 'NDWI', 'EVI',
]
INTERNAL_CLASS_TO_ID = {
    'Built-up/Bare soil': 0,
    'Durian': 1,
    'Forest': 2,
    'Mixed agriculture': 3,
    'Oil palm': 4,
    'Rubber': 5,
    'Water': 6,
}
REPORTING_ID_TO_CLASS = {
    0: 'Built-up/Bare soil',
    1: 'Durian',
    2: 'Forest',
    3: 'Other agriculture',
    4: 'Oil palm',
    5: 'Rubber',
    6: 'Water',
}
VALID_SOURCE_CLASS_TO_ID = {
    **INTERNAL_CLASS_TO_ID,
    'Other agriculture': 3,
}

required_paths = [INPUT_CSV, MODEL_BUNDLE_PATH]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        'Required E3 input(s) are missing:\n' + '\n'.join(missing_paths)
    )

for folder in [OUTPUT_DIR, TABLE_DIR, FIGURE_DIR, METADATA_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

if sklearn.__version__ != EXPECTED_SKLEARN_VERSION:
    warnings.warn(
        f'Original E3 used scikit-learn {EXPECTED_SKLEARN_VERSION}; '
        f'current version is {sklearn.__version__}. The locked-result '
        'gate will detect any numerical change.',
        RuntimeWarning,
    )

print('Input CSV:', INPUT_CSV)
print('Frozen Bentong model:', MODEL_BUNDLE_PATH)
print('E2 summary available:', E2_SUMMARY_PATH.exists())
print('New E3 output:', OUTPUT_DIR)


## 3. Reusable audit and atomic-output functions


In [ ]:
def sha256_file(path, chunk_size=16 * 1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as stream:
        while True:
            block = stream.read(chunk_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()


def atomic_write_csv(frame, path, index=False):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    frame.to_csv(temporary, index=index)
    os.replace(temporary, path)


def atomic_write_json(payload, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    with open(temporary, 'w', encoding='utf-8') as stream:
        json.dump(payload, stream, indent=2, ensure_ascii=False)
    os.replace(temporary, path)


def atomic_write_text(text, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    temporary.write_text(text, encoding='utf-8')
    os.replace(temporary, path)


def stable_seed(text, base_seed):
    digest = hashlib.sha256(str(text).encode('utf-8')).hexdigest()
    return (int(digest[:8], 16) + int(base_seed)) % (2**32 - 1)


input_csv_hash = sha256_file(INPUT_CSV)
model_bundle_hash = sha256_file(MODEL_BUNDLE_PATH)
print('Input CSV SHA-256:', input_csv_hash)
print('Model bundle SHA-256:', model_bundle_hash)

if VERIFY_INPUT_HASHES:
    if input_csv_hash != EXPECTED_INPUT_CSV_SHA256:
        raise AssertionError(
            'Pahang input CSV is not the locked E3 version. '
            f'Expected {EXPECTED_INPUT_CSV_SHA256}, found {input_csv_hash}.'
        )
    if model_bundle_hash != EXPECTED_MODEL_BUNDLE_SHA256:
        raise AssertionError(
            'Frozen Bentong model is not the locked E3 model. '
            f'Expected {EXPECTED_MODEL_BUNDLE_SHA256}, '
            f'found {model_bundle_hash}.'
        )
print('Locked input hashes passed.')


## 4. Load and audit the frozen Bentong model

This cell only loads an already fitted model. It locks the selected
features, candidate and class encoding before any Pahang data are read.


In [ ]:
model_bundle = joblib.load(MODEL_BUNDLE_PATH)
if not isinstance(model_bundle, dict) or 'model' not in model_bundle:
    raise TypeError('The joblib bundle must be a dict containing "model".')

frozen_model = model_bundle['model']
selected_features = [
    str(feature)
    for feature in model_bundle.get('predictor_bands', EXPECTED_S2_FEATURES)
]
if selected_features != EXPECTED_S2_FEATURES:
    raise AssertionError(
        f'Expected locked S2 predictors {EXPECTED_S2_FEATURES}; '
        f'found {selected_features}.'
    )

class_to_id = {
    str(name): int(class_id)
    for name, class_id in model_bundle.get(
        'class_to_id', INTERNAL_CLASS_TO_ID
    ).items()
}
if class_to_id != INTERNAL_CLASS_TO_ID:
    raise AssertionError(
        f'Frozen model class mapping changed: {class_to_id}'
    )

feature_set = str(model_bundle.get('feature_set'))
candidate_id = str(model_bundle.get('candidate_id'))
svm_params = model_bundle.get('svm_params', {})
expected_params = {
    'candidate_id': 'R1',
    'model_type': 'rbf_svc',
    'C': 10.0,
    'gamma': 'scale',
    'tol': 0.001,
}
if feature_set != 'S2' or candidate_id != 'R1':
    raise AssertionError(
        f'Expected frozen S2 + R1; found {feature_set} + {candidate_id}.'
    )
for key, expected in expected_params.items():
    if svm_params.get(key) != expected:
        raise AssertionError(
            f'Frozen R1 parameter {key} changed: '
            f'{svm_params.get(key)!r} != {expected!r}'
        )

classifier = (
    frozen_model.named_steps.get('classifier')
    if hasattr(frozen_model, 'named_steps') else None
)
if classifier is None or not hasattr(classifier, 'classes_'):
    raise TypeError('Expected a fitted sklearn pipeline and classifier.')

class_ids = sorted(REPORTING_ID_TO_CLASS)
model_class_ids = [int(value) for value in classifier.classes_]
if model_class_ids != class_ids:
    raise AssertionError(
        f'Model class order changed: {model_class_ids} != {class_ids}'
    )

random_seed = int(model_bundle.get('random_seed', EXPECTED_RANDOM_SEED))
if random_seed != EXPECTED_RANDOM_SEED:
    raise AssertionError(
        f'Random seed changed: {random_seed} != {EXPECTED_RANDOM_SEED}'
    )

report_class_names = [REPORTING_ID_TO_CLASS[i] for i in class_ids]
durian_id = INTERNAL_CLASS_TO_ID['Durian']
rubber_id = INTERNAL_CLASS_TO_ID['Rubber']

print('Model is fitted:', hasattr(classifier, 'support_'))
print('Frozen feature set / candidate:', feature_set, candidate_id)
print('Frozen SVM parameters:', svm_params)
print('Predictors:', selected_features)
print('Internal class mapping:', class_to_id)
print('Reporting class mapping:', REPORTING_ID_TO_CLASS)


## 5. Read and audit the independent Pahang CSV


In [ ]:
required_metadata = [
    'pixel_uid', 'sample_uid', 'group_uid', 'class_lv2', 'label_id'
]
optional_metadata = [
    'class_id', 'spatial_group_no', 's_uuid', 'domain',
    'longitude', 'latitude', 'valid_count',
]

raw_df = pd.read_csv(INPUT_CSV, low_memory=False)
missing_columns = sorted(
    set(required_metadata + selected_features) - set(raw_df.columns)
)
if missing_columns:
    raise ValueError(f'Missing required CSV columns: {missing_columns}')

df = raw_df.copy()
for column in ['pixel_uid', 'sample_uid', 'group_uid', 'class_lv2']:
    df[column] = df[column].astype('string').str.strip()
for column in ['class_id', 'spatial_group_no', 's_uuid', 'domain']:
    if column in df.columns:
        df[column] = df[column].astype('string').str.strip()

df['label_id'] = pd.to_numeric(df['label_id'], errors='coerce')
for feature in selected_features:
    df[feature] = pd.to_numeric(
        df[feature], errors='coerce'
    ).astype('float32')
for column in ['longitude', 'latitude', 'valid_count']:
    if column in df.columns:
        df[column] = pd.to_numeric(df[column], errors='coerce')

df = df.replace([np.inf, -np.inf], np.nan)
invalid_required_string = pd.Series(False, index=df.index)
for column in ['pixel_uid', 'sample_uid', 'group_uid', 'class_lv2']:
    invalid_required_string |= df[column].isna() | df[column].eq('')
invalid_numeric = df[selected_features + ['label_id']].isna().any(axis=1)
invalid_mask = invalid_required_string | invalid_numeric

if invalid_mask.any():
    invalid_rows = df.loc[
        invalid_mask,
        ['pixel_uid', 'sample_uid', 'group_uid', 'class_lv2', 'label_id'],
    ]
    atomic_write_csv(
        invalid_rows,
        TABLE_DIR / 'dropped_rows_missing_required_values.csv',
    )
    print('Dropped invalid rows:', int(invalid_mask.sum()))
df = df.loc[~invalid_mask].copy()
df['label_id'] = df['label_id'].astype('int16')

if df['pixel_uid'].duplicated().any():
    raise ValueError(
        f'pixel_uid has {int(df["pixel_uid"].duplicated().sum())} duplicates.'
    )

expected_label_id = df['class_lv2'].map(VALID_SOURCE_CLASS_TO_ID)
mismatch_mask = expected_label_id.isna() | (
    expected_label_id.astype('Int16') != df['label_id'].astype('Int16')
)
if mismatch_mask.any():
    mismatches = df.loc[
        mismatch_mask, ['sample_uid', 'class_lv2', 'label_id']
    ].drop_duplicates()
    atomic_write_csv(mismatches, TABLE_DIR / 'class_label_mismatches.csv')
    raise ValueError(
        f'class_lv2 and label_id mismatch in {int(mismatch_mask.sum())} rows.'
    )

unknown_labels = sorted(set(df['label_id'].astype(int)) - set(class_ids))
if unknown_labels:
    raise ValueError(f'Unknown label_id values: {unknown_labels}')

df['report_class_lv2'] = (
    df['label_id'].astype(int).map(REPORTING_ID_TO_CLASS)
)
if 'domain' in df.columns:
    domains = sorted(df['domain'].dropna().unique().tolist())
    if domains != ['Pahang_without_Bentong']:
        raise AssertionError(
            'E3 must contain only Pahang outside Bentong; '
            f'found domains {domains}.'
        )

print(f'Raw CSV rows: {len(raw_df):,}')
print(f'Usable rows: {len(df):,}')
print('Polygons:', df['sample_uid'].nunique())
print('Groups:', df['group_uid'].nunique())


## 6. Polygon and spatial-group consistency checks


In [ ]:
sample_consistency = (
    df.groupby('sample_uid')
    .agg(
        pixels_raw=('pixel_uid', 'size'),
        class_nunique=('label_id', 'nunique'),
        group_nunique=('group_uid', 'nunique'),
        label_id=('label_id', 'first'),
        group_uid=('group_uid', 'first'),
        class_lv2=('report_class_lv2', 'first'),
    )
    .reset_index()
)
bad_samples = sample_consistency.loc[
    (sample_consistency['class_nunique'] != 1)
    | (sample_consistency['group_nunique'] != 1)
]
if len(bad_samples):
    atomic_write_csv(bad_samples, TABLE_DIR / 'bad_sample_consistency.csv')
    raise AssertionError(
        f'{len(bad_samples)} sample_uid values contain mixed labels/groups.'
    )

group_consistency = (
    sample_consistency.groupby('group_uid')
    .agg(
        polygon_count=('sample_uid', 'size'),
        class_nunique=('label_id', 'nunique'),
        label_id=('label_id', 'first'),
        class_lv2=('class_lv2', 'first'),
    )
    .reset_index()
)
bad_groups = group_consistency.loc[group_consistency['class_nunique'] != 1]
if len(bad_groups):
    atomic_write_csv(bad_groups, TABLE_DIR / 'bad_group_consistency.csv')
    raise AssertionError(
        f'{len(bad_groups)} group_uid values contain mixed classes.'
    )

raw_class_summary = (
    df.groupby(['label_id', 'report_class_lv2'])
    .agg(
        pixel_rows=('pixel_uid', 'size'),
        polygon_count=('sample_uid', 'nunique'),
        group_count=('group_uid', 'nunique'),
    )
    .reset_index()
    .rename(columns={'report_class_lv2': 'class_lv2'})
    .sort_values('label_id')
)
atomic_write_csv(raw_class_summary, TABLE_DIR / 'pahang_raw_class_summary.csv')
atomic_write_csv(
    sample_consistency, TABLE_DIR / 'pahang_raw_sample_pixel_counts.csv'
)
atomic_write_csv(group_consistency, TABLE_DIR / 'pahang_group_summary.csv')

print('Sample consistency: passed')
print('Group consistency: passed')
display(raw_class_summary)


## 7. Reproduce the evaluation sampling and equal-polygon weights

At most 100 pixels are retained per polygon using the original stable
SHA-based seed. Each retained polygon receives total pixel weight 1, so a
large polygon cannot dominate pixel-level accuracy.


In [ ]:
retained_parts = []
for sample_uid, group in df.groupby('sample_uid', sort=True):
    group = group.sort_values('pixel_uid')
    if len(group) > MAX_PIXELS_PER_SAMPLE:
        group = group.sample(
            n=MAX_PIXELS_PER_SAMPLE,
            random_state=stable_seed(sample_uid, random_seed),
            replace=False,
        )
    retained_parts.append(group)

model_df = (
    pd.concat(retained_parts, ignore_index=True)
    .sort_values('pixel_uid')
    .reset_index(drop=True)
)
retained_counts = model_df.groupby('sample_uid')['pixel_uid'].transform('size')
model_df['sample_weight'] = (1.0 / retained_counts).astype('float32')

polygon_weight_sums = model_df.groupby('sample_uid')['sample_weight'].sum()
if not np.allclose(polygon_weight_sums.to_numpy(), 1.0):
    raise AssertionError('Each polygon must have total sample weight 1.')

retained_sample_summary = (
    model_df.groupby('sample_uid')
    .agg(
        pixels_retained=('pixel_uid', 'size'),
        sample_weight_sum=('sample_weight', 'sum'),
        label_id=('label_id', 'first'),
        class_lv2=('report_class_lv2', 'first'),
        group_uid=('group_uid', 'first'),
    )
    .reset_index()
)
retained_class_summary = (
    model_df.groupby(['label_id', 'report_class_lv2'])
    .agg(
        pixel_rows=('pixel_uid', 'size'),
        polygon_count=('sample_uid', 'nunique'),
        group_count=('group_uid', 'nunique'),
        weight_sum=('sample_weight', 'sum'),
    )
    .reset_index()
    .rename(columns={'report_class_lv2': 'class_lv2'})
    .sort_values('label_id')
)
atomic_write_csv(
    retained_sample_summary,
    TABLE_DIR / 'pahang_model_sample_pixel_counts.csv',
)
atomic_write_csv(
    retained_class_summary,
    TABLE_DIR / 'pahang_model_class_summary.csv',
)

observed_counts = {
    'raw_rows': int(len(df)),
    'retained_rows': int(len(model_df)),
    'samples': int(model_df['sample_uid'].nunique()),
    'groups': int(model_df['group_uid'].nunique()),
}
expected_counts = {
    'raw_rows': EXPECTED_RAW_ROWS,
    'retained_rows': EXPECTED_RETAINED_ROWS,
    'samples': EXPECTED_SAMPLE_COUNT,
    'groups': EXPECTED_GROUP_COUNT,
}
if observed_counts != expected_counts:
    raise AssertionError(
        f'E3 retained-data counts changed: {observed_counts} '
        f'!= {expected_counts}'
    )
print('Locked row/polygon/group counts passed:', observed_counts)
display(retained_class_summary)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.barplot(
    data=retained_class_summary,
    x='class_lv2', y='pixel_rows', ax=axes[0], color='#3B82F6',
)
axes[0].set_title('E3 retained pixel rows by class')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=45)

sns.boxplot(
    data=retained_sample_summary,
    x='class_lv2', y='pixels_retained', ax=axes[1], color='#A7F3D0',
)
axes[1].set_title('E3 retained pixels per polygon')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(
    FIGURE_DIR / 'pahang_retained_pixel_distribution.png',
    dpi=180, bbox_inches='tight',
)
plt.show()


## 8. Predict Pahang pixels with the frozen Bentong model

This is the only model-use step. It calls `predict()` and
`decision_function()` only. There is deliberately no `.fit()` operation.


In [ ]:
X_external = model_df[selected_features].astype('float32')
prediction_start = time.time()
pixel_pred_class_id = frozen_model.predict(X_external).astype('int16')
decision_scores = frozen_model.decision_function(X_external)
print('Prediction runtime (seconds):', round(time.time() - prediction_start, 1))

if decision_scores.ndim != 2:
    raise ValueError('Expected a 2-D multiclass decision-score array.')
if decision_scores.shape != (len(model_df), len(model_class_ids)):
    raise ValueError(
        f'Unexpected decision-score shape: {decision_scores.shape}'
    )

score_columns = [f'decision_score_{i}' for i in model_class_ids]
metadata_columns = [
    column for column in [
        'pixel_uid', 'sample_uid', 'group_uid', 'label_id',
        'class_id', 'spatial_group_no', 's_uuid', 'domain',
        'longitude', 'latitude', 'valid_count', 'sample_weight',
    ] if column in model_df.columns
]
pixel_predictions = model_df[metadata_columns].copy().rename(
    columns={'label_id': 'true_class_id'}
)
pixel_predictions['true_class_name'] = (
    pixel_predictions['true_class_id'].astype(int).map(REPORTING_ID_TO_CLASS)
)
pixel_predictions['pred_class_id'] = pixel_pred_class_id
pixel_predictions['pred_class_name'] = (
    pixel_predictions['pred_class_id'].astype(int).map(REPORTING_ID_TO_CLASS)
)
score_frame = pd.DataFrame(
    decision_scores,
    columns=score_columns,
    index=pixel_predictions.index,
).astype('float32')
pixel_predictions = pd.concat([pixel_predictions, score_frame], axis=1)
atomic_write_csv(
    pixel_predictions, TABLE_DIR / 'pahang_pixel_predictions.csv'
)
print('Pixel prediction table:', pixel_predictions.shape)
display(pixel_predictions.head())


## 9. Aggregate decision scores to polygon predictions


In [ ]:
def aggregate_polygon_predictions(pixel_frame):
    first_columns = [
        column for column in [
            'sample_uid', 'group_uid', 'true_class_id', 'true_class_name',
            'class_id', 'spatial_group_no', 's_uuid', 'domain',
        ] if column in pixel_frame.columns
    ]
    first_part = (
        pixel_frame[first_columns]
        .groupby('sample_uid', as_index=False)
        .first()
    )
    score_part = (
        pixel_frame[['sample_uid'] + score_columns]
        .groupby('sample_uid', as_index=False)
        .mean()
    )
    count_part = (
        pixel_frame.groupby('sample_uid')
        .agg(
            retained_pixel_count=('pixel_uid', 'size'),
            sample_weight_sum=('sample_weight', 'sum'),
        )
        .reset_index()
    )
    result = (
        first_part
        .merge(score_part, on='sample_uid', how='left', validate='one_to_one')
        .merge(count_part, on='sample_uid', how='left', validate='one_to_one')
    )
    score_matrix = result[score_columns].to_numpy()
    result['pred_class_id'] = np.asarray(model_class_ids)[
        score_matrix.argmax(axis=1)
    ].astype('int16')
    result['pred_class_name'] = (
        result['pred_class_id'].astype(int).map(REPORTING_ID_TO_CLASS)
    )
    result['correct'] = (
        result['pred_class_id'].astype(int)
        == result['true_class_id'].astype(int)
    )
    return result


polygon_predictions = aggregate_polygon_predictions(pixel_predictions)
if polygon_predictions['sample_uid'].duplicated().any():
    raise AssertionError('Polygon prediction table contains duplicate sample_uid.')
if len(polygon_predictions) != EXPECTED_SAMPLE_COUNT:
    raise AssertionError('Polygon prediction table is incomplete.')
atomic_write_csv(
    polygon_predictions, TABLE_DIR / 'pahang_polygon_predictions.csv'
)
print(
    'Correct polygons:', int(polygon_predictions['correct'].sum()),
    '/', len(polygon_predictions),
)
display(polygon_predictions.head())


## 10. Calculate external validation metrics


In [ ]:
def metric_dictionary(y_true, y_pred, sample_weight=None):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)

    durian_prf = precision_recall_fscore_support(
        y_true, y_pred, labels=[durian_id], average=None,
        sample_weight=sample_weight, zero_division=0,
    )
    rubber_prf = precision_recall_fscore_support(
        y_true, y_pred, labels=[rubber_id], average=None,
        sample_weight=sample_weight, zero_division=0,
    )
    return {
        'accuracy': float(accuracy_score(y_true, y_pred, sample_weight=sample_weight)),
        'balanced_accuracy': float(
            balanced_accuracy_score(y_true, y_pred, sample_weight=sample_weight)
        ),
        'macro_f1': float(
            f1_score(
                y_true, y_pred, labels=class_ids, average='macro',
                sample_weight=sample_weight, zero_division=0,
            )
        ),
        'durian_precision': float(durian_prf[0][0]),
        'durian_recall': float(durian_prf[1][0]),
        'durian_f1': float(durian_prf[2][0]),
        'rubber_precision': float(rubber_prf[0][0]),
        'rubber_recall': float(rubber_prf[1][0]),
        'rubber_f1': float(rubber_prf[2][0]),
        'support_weighted_sum': float(
            np.sum(sample_weight) if sample_weight is not None else len(y_true)
        ),
    }


def report_frame(y_true, y_pred, sample_weight=None):
    report = classification_report(
        np.asarray(y_true).astype(int),
        np.asarray(y_pred).astype(int),
        labels=class_ids,
        target_names=report_class_names,
        sample_weight=sample_weight,
        output_dict=True,
        zero_division=0,
    )
    return pd.DataFrame(report).T


pixel_metrics = metric_dictionary(
    pixel_predictions['true_class_id'],
    pixel_predictions['pred_class_id'],
    sample_weight=pixel_predictions['sample_weight'],
)
polygon_metrics = metric_dictionary(
    polygon_predictions['true_class_id'],
    polygon_predictions['pred_class_id'],
)
external_metrics = {
    **{f'pixel_{key}': value for key, value in pixel_metrics.items()},
    **{f'polygon_{key}': value for key, value in polygon_metrics.items()},
}
external_metrics_table = pd.DataFrame([
    {'level': 'pixel_weighted', **pixel_metrics},
    {'level': 'polygon', **polygon_metrics},
])
pixel_report = report_frame(
    pixel_predictions['true_class_id'],
    pixel_predictions['pred_class_id'],
    sample_weight=pixel_predictions['sample_weight'],
)
polygon_report = report_frame(
    polygon_predictions['true_class_id'],
    polygon_predictions['pred_class_id'],
)
atomic_write_csv(
    external_metrics_table, TABLE_DIR / 'pahang_external_metrics.csv'
)
atomic_write_csv(
    pixel_report, TABLE_DIR / 'pahang_pixel_classification_report.csv',
    index=True,
)
atomic_write_csv(
    polygon_report, TABLE_DIR / 'pahang_polygon_classification_report.csv',
    index=True,
)
display(external_metrics_table)
display(polygon_report)


## 11. Confusion matrices and locked E3 reproduction gate


In [ ]:
def save_confusion_outputs(
    y_true, y_pred, title, stem, sample_weight=None
):
    matrix = confusion_matrix(
        np.asarray(y_true).astype(int),
        np.asarray(y_pred).astype(int),
        labels=class_ids,
        sample_weight=sample_weight,
    )
    frame = pd.DataFrame(
        matrix, index=report_class_names, columns=report_class_names
    )
    atomic_write_csv(frame, TABLE_DIR / f'{stem}.csv', index=True)
    plt.figure(figsize=(9, 7))
    sns.heatmap(
        frame, annot=True,
        fmt='.2f' if sample_weight is not None else 'd',
        cmap='Blues', linewidths=0.5, linecolor='white',
    )
    plt.title(title)
    plt.xlabel('Predicted class')
    plt.ylabel('Reference class')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / f'{stem}.png', dpi=180, bbox_inches='tight')
    plt.show()
    return frame


pixel_confusion = save_confusion_outputs(
    pixel_predictions['true_class_id'],
    pixel_predictions['pred_class_id'],
    'E3 weighted pixel confusion matrix',
    'pahang_pixel_confusion_matrix',
    sample_weight=pixel_predictions['sample_weight'],
)
polygon_confusion = save_confusion_outputs(
    polygon_predictions['true_class_id'],
    polygon_predictions['pred_class_id'],
    'E3 polygon confusion matrix',
    'pahang_polygon_confusion_matrix',
)

LOCKED_E3_METRICS = {
    'pixel_accuracy': 0.64263209317431,
    'pixel_balanced_accuracy': 0.6417489465042857,
    'pixel_macro_f1': 0.6462825036229921,
    'pixel_durian_precision': 0.575596164131163,
    'pixel_durian_recall': 0.6490507687656055,
    'pixel_durian_f1': 0.6101205544102021,
    'polygon_accuracy': 0.7039877300613497,
    'polygon_balanced_accuracy': 0.7046642061106559,
    'polygon_macro_f1': 0.7037564048022223,
    'polygon_durian_precision': 0.6923076923076923,
    'polygon_durian_recall': 0.7431192660550459,
    'polygon_durian_f1': 0.7168141592920354,
}
LOCKED_E3_POLYGON_CONFUSION = np.asarray([
    [75, 0, 3, 5, 0, 1, 3],
    [1, 81, 6, 1, 2, 18, 0],
    [1, 2, 76, 2, 1, 16, 0],
    [9, 6, 3, 39, 0, 0, 4],
    [0, 5, 10, 1, 64, 24, 1],
    [0, 19, 26, 13, 3, 37, 0],
    [2, 4, 0, 1, 0, 0, 87],
], dtype='int64')

reproduction_records = []
for metric_name, expected_value in LOCKED_E3_METRICS.items():
    observed_value = float(external_metrics[metric_name])
    difference = abs(observed_value - expected_value)
    reproduction_records.append({
        'check': metric_name,
        'expected': expected_value,
        'observed': observed_value,
        'absolute_difference': difference,
        'passed': bool(np.isclose(
            observed_value, expected_value,
            rtol=0, atol=RESULT_ABSOLUTE_TOLERANCE,
        )),
    })

polygon_confusion_array = polygon_confusion.to_numpy(dtype='int64')
confusion_passed = np.array_equal(
    polygon_confusion_array, LOCKED_E3_POLYGON_CONFUSION
)
reproduction_records.append({
    'check': 'polygon_confusion_matrix_exact',
    'expected': 'locked 7x7 integer matrix',
    'observed': 'reproduced 7x7 integer matrix',
    'absolute_difference': int(np.abs(
        polygon_confusion_array - LOCKED_E3_POLYGON_CONFUSION
    ).sum()),
    'passed': bool(confusion_passed),
})
reproduction_check = pd.DataFrame(reproduction_records)
all_locked_e3_checks_passed = bool(reproduction_check['passed'].all())
atomic_write_csv(
    reproduction_check,
    TABLE_DIR / 'E3_locked_result_reproduction_check.csv',
)
display(reproduction_check)
print('ALL LOCKED E3 CHECKS PASSED:', all_locked_e3_checks_passed)
if REQUIRE_LOCKED_RESULT_MATCH and not all_locked_e3_checks_passed:
    failed = reproduction_check.loc[
        ~reproduction_check['passed'], 'check'
    ].tolist()
    raise AssertionError(
        'E3 reconstruction differs from locked results: ' + ', '.join(failed)
    )


## 12. Matched descriptive comparison with E2

E2 is the fixed S2+R1 grouped OOF Bentong benchmark. E3 uses the frozen
all-Bentong model on independent Pahang data. Their configuration is
matched, but their evaluation roles differ; the differences below are
descriptive generalization differences, not causal estimates.


In [ ]:
LOCKED_E2_METRICS = {
    'pixel_accuracy': 0.7994308298113575,
    'pixel_balanced_accuracy': 0.7947360904868594,
    'pixel_macro_f1': 0.7727588701330619,
    'pixel_durian_f1': 0.8052719884723906,
    'pixel_rubber_f1': 0.5891683410626712,
    'polygon_accuracy': 0.8545780969479354,
    'polygon_balanced_accuracy': 0.8483189902467012,
    'polygon_macro_f1': 0.8315074621606635,
    'polygon_durian_precision': 0.8412698412698413,
    'polygon_durian_recall': 0.8833333333333333,
    'polygon_durian_f1': 0.8617886178861789,
    'polygon_rubber_f1': 0.6923076923076923,
}
e2_metric_source = 'embedded locked E2 values'
e2_metrics = dict(LOCKED_E2_METRICS)

if E2_SUMMARY_PATH.exists():
    e2_summary = pd.read_csv(E2_SUMMARY_PATH)
    if len(e2_summary) != 1:
        raise AssertionError('E2 summary must contain exactly one row.')
    e2_row = e2_summary.iloc[0]
    e2_column_map = {
        'pixel_accuracy': 'pooled_pixel_accuracy',
        'pixel_balanced_accuracy': 'pooled_pixel_balanced_accuracy',
        'pixel_macro_f1': 'pooled_pixel_macro_f1',
        'pixel_durian_f1': 'pooled_pixel_durian_f1',
        'pixel_rubber_f1': 'pooled_pixel_rubber_f1',
        'polygon_accuracy': 'pooled_polygon_accuracy',
        'polygon_balanced_accuracy': 'pooled_polygon_balanced_accuracy',
        'polygon_macro_f1': 'pooled_polygon_macro_f1',
        'polygon_durian_precision': 'pooled_polygon_durian_precision',
        'polygon_durian_recall': 'pooled_polygon_durian_recall',
        'polygon_durian_f1': 'pooled_polygon_durian_f1',
        'polygon_rubber_f1': 'pooled_polygon_rubber_f1',
    }
    for short_name, source_column in e2_column_map.items():
        observed = float(e2_row[source_column])
        expected = LOCKED_E2_METRICS[short_name]
        if not np.isclose(observed, expected, rtol=0, atol=1e-9):
            raise AssertionError(
                f'E2 metric {source_column} is not the locked value.'
            )
        e2_metrics[short_name] = observed
    e2_metric_source = str(E2_SUMMARY_PATH)

comparison_metrics = [
    'pixel_accuracy', 'pixel_balanced_accuracy', 'pixel_macro_f1',
    'pixel_durian_f1', 'pixel_rubber_f1',
    'polygon_accuracy', 'polygon_balanced_accuracy', 'polygon_macro_f1',
    'polygon_durian_precision', 'polygon_durian_recall',
    'polygon_durian_f1', 'polygon_rubber_f1',
]
comparison_records = []
for metric_name in comparison_metrics:
    e2_value = float(e2_metrics[metric_name])
    e3_value = float(external_metrics[metric_name])
    comparison_records.append({
        'metric': metric_name,
        'E2_Bentong_fixed_grouped_OOF': e2_value,
        'E3_Pahang_external': e3_value,
        'E3_minus_E2_descriptive_difference': e3_value - e2_value,
    })
e2_vs_e3 = pd.DataFrame(comparison_records)
atomic_write_csv(e2_vs_e3, TABLE_DIR / 'E2_vs_E3_matched_metrics.csv')
print('E2 metric source:', e2_metric_source)
display(e2_vs_e3)

plot_metrics = [
    'polygon_accuracy', 'polygon_macro_f1',
    'polygon_durian_f1', 'polygon_rubber_f1',
]
plot_frame = e2_vs_e3.loc[
    e2_vs_e3['metric'].isin(plot_metrics)
].melt(
    id_vars='metric',
    value_vars=[
        'E2_Bentong_fixed_grouped_OOF', 'E3_Pahang_external'
    ],
    var_name='evaluation', value_name='score',
)
plt.figure(figsize=(11, 5))
sns.barplot(data=plot_frame, x='metric', y='score', hue='evaluation')
plt.ylim(0, 1)
plt.title('Matched S2+R1 comparison: E2 Bentong vs E3 Pahang external')
plt.xlabel('')
plt.ylabel('Score')
plt.xticks(rotation=25, ha='right')
plt.tight_layout()
plt.savefig(
    FIGURE_DIR / 'E2_vs_E3_matched_polygon_metrics.png',
    dpi=180, bbox_inches='tight',
)
plt.show()


## 13. Save manifest, README and completion gate


In [ ]:
output_paths = {
    'external_metrics': TABLE_DIR / 'pahang_external_metrics.csv',
    'pixel_predictions': TABLE_DIR / 'pahang_pixel_predictions.csv',
    'polygon_predictions': TABLE_DIR / 'pahang_polygon_predictions.csv',
    'pixel_classification_report':
        TABLE_DIR / 'pahang_pixel_classification_report.csv',
    'polygon_classification_report':
        TABLE_DIR / 'pahang_polygon_classification_report.csv',
    'pixel_confusion': TABLE_DIR / 'pahang_pixel_confusion_matrix.csv',
    'polygon_confusion': TABLE_DIR / 'pahang_polygon_confusion_matrix.csv',
    'locked_reproduction_check':
        TABLE_DIR / 'E3_locked_result_reproduction_check.csv',
    'E2_vs_E3_comparison': TABLE_DIR / 'E2_vs_E3_matched_metrics.csv',
}
missing_outputs = [
    str(path) for path in output_paths.values() if not path.exists()
]
if missing_outputs:
    raise AssertionError(
        'Required E3 output(s) are missing:\n' + '\n'.join(missing_outputs)
    )
output_hashes = {
    name: sha256_file(path) for name, path in output_paths.items()
}

manifest = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'experiment': 'E3',
    'experiment_definition': (
        'Frozen all-Bentong S2+R1 SVM applied to independently '
        'labelled Pahang outside Bentong'
    ),
    'analytical_status': 'INDEPENDENT_EXTERNAL_EVALUATION',
    'pahang_used_for_training': False,
    'pahang_used_for_feature_selection': False,
    'pahang_used_for_hyperparameter_tuning': False,
    'pahang_labels_used_only_for_accuracy_evaluation': True,
    'input_csv': str(INPUT_CSV),
    'input_csv_sha256': input_csv_hash,
    'model_bundle': str(MODEL_BUNDLE_PATH),
    'model_bundle_sha256': model_bundle_hash,
    'model': {
        'model_type': 'SVM',
        'feature_set': feature_set,
        'candidate_id': candidate_id,
        'svm_params': svm_params,
        'predictor_bands': selected_features,
        'probability_calibration': bool(
            model_bundle.get('probability_calibration', False)
        ),
    },
    'evaluation': {
        'max_pixels_per_polygon': MAX_PIXELS_PER_SAMPLE,
        'random_seed': random_seed,
        'pixel_weighting': 'each polygon has total pixel weight 1',
        'polygon_aggregation': 'mean multiclass decision score',
        'raw_rows': int(len(df)),
        'retained_rows': int(len(model_df)),
        'polygon_count': int(model_df['sample_uid'].nunique()),
        'group_count': int(model_df['group_uid'].nunique()),
    },
    'internal_class_to_id': INTERNAL_CLASS_TO_ID,
    'reporting_id_to_class': REPORTING_ID_TO_CLASS,
    'class_3_note': (
        'Internal Mixed agriculture remains class 3; thesis outputs '
        'report it as Other agriculture.'
    ),
    'external_metrics': external_metrics,
    'E2_metric_source': e2_metric_source,
    'all_locked_E3_checks_passed': all_locked_e3_checks_passed,
    'output_sha256': output_hashes,
    'software': {
        'python': sys.version,
        'platform': platform.platform(),
        'numpy': np.__version__,
        'pandas': pd.__version__,
        'scikit_learn': sklearn.__version__,
        'joblib': joblib.__version__,
    },
}
atomic_write_json(manifest, METADATA_DIR / 'E3_run_manifest.json')

readme_text = f"""E3 frozen Bentong S2+R1 model applied to Pahang outside Bentong
=====================================================================

Analytical status: independent external evaluation.

Pahang labels were not used for training, feature selection, tuning,
threshold adjustment or class merging. The frozen Bentong model hash is:
{model_bundle_hash}

Raw/retained pixels: {len(df):,} / {len(model_df):,}
Polygons/groups: {model_df['sample_uid'].nunique()} / {model_df['group_uid'].nunique()}

Class 3 is stored internally as Mixed agriculture and reported in thesis
outputs as Other agriculture without changing its numeric encoding.

Inspect tables/E3_locked_result_reproduction_check.csv before using the
reconstructed outputs. E2_vs_E3_matched_metrics.csv is a descriptive
configuration-matched comparison, not a causal domain-shift estimate.
"""
atomic_write_text(readme_text, OUTPUT_DIR / 'README_E3.txt')

completion_status = {
    'experiment': 'E3',
    'input_hashes_locked': bool(
        input_csv_hash == EXPECTED_INPUT_CSV_SHA256
        and model_bundle_hash == EXPECTED_MODEL_BUNDLE_SHA256
    ),
    'rows_complete': len(model_df) == EXPECTED_RETAINED_ROWS,
    'polygons_complete': (
        model_df['sample_uid'].nunique() == EXPECTED_SAMPLE_COUNT
    ),
    'groups_complete': (
        model_df['group_uid'].nunique() == EXPECTED_GROUP_COUNT
    ),
    'model_locked': bool(
        feature_set == 'S2' and candidate_id == 'R1'
    ),
    'no_pahang_training': True,
    'all_locked_result_checks_passed': all_locked_e3_checks_passed,
}
completion_status['ready_for_thesis_E3_use'] = bool(all(
    value for key, value in completion_status.items()
    if key != 'experiment'
))
atomic_write_json(
    completion_status, METADATA_DIR / 'E3_completion_status.json'
)
if not completion_status['ready_for_thesis_E3_use']:
    raise AssertionError('E3 completion gate did not pass.')
atomic_write_text(
    datetime.now(timezone.utc).isoformat() + '\n',
    METADATA_DIR / 'COMPLETED.txt',
)

print(json.dumps(completion_status, indent=2, ensure_ascii=False))
print('E3 reconstruction completed:', OUTPUT_DIR)
